In [3]:
import os

original_dir = "/home/takudzwa/Documents/Projects/broiler_disease/detection/FOMO/dataset/train/others"
augmented_dir = "/home/takudzwa/Documents/Projects/broiler_disease/detection/FOMO/dataset/train/aug_other"

os.makedirs(augmented_dir, exist_ok=True)


In [5]:
from PIL import Image, ImageEnhance, ImageOps
import random

def augment_image(image):
    # Random rotation
    angle = random.choice([0, 90, 180, 270])
    image = image.rotate(angle)

    # Random horizontal flip
    if random.random() > 0.5:
        image = ImageOps.mirror(image)

    # Random brightness
    enhancer = ImageEnhance.Brightness(image)
    image = enhancer.enhance(random.uniform(0.7, 1.3))  # brighter/darker

    return image


In [7]:
import os
from PIL import Image

target_count = 1200

# Get original image files
image_files = [f for f in os.listdir(original_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
current_count = len(image_files)

# Augment until reaching 1200
i = 1
while i + current_count <= target_count:
    img_name = random.choice(image_files)
    img_path = os.path.join(original_dir, img_name)
    
    try:
        img = Image.open(img_path)
        aug_img = augment_image(img).convert("RGB")
        aug_img.save(os.path.join(augmented_dir, f"other_aug_{i}.jpg"))
        i += 1
    except Exception as e:
        print(f"Error with {img_name}: {e}")


In [3]:
import os
import cv2
import albumentations as A
import random
from tqdm import tqdm

# Paths
INPUT_DIR = '/home/takudzwa/Documents/Projects/broiler_disease/detection/FOMO/dataset/train/healthy'
OUTPUT_DIR = '/home/takudzwa/Documents/Projects/broiler_disease/detection/FOMO/dataset/train/ncd_aug'

TARGET_COUNT = 1250

# Create output directory if not exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load existing images
image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
current_count = len(image_files)
needed = TARGET_COUNT - current_count

# Define augmentation pipeline
augmentations = A.Compose([
    A.RandomBrightnessContrast(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomCrop(width=200, height=200, p=0.5),
    A.Blur(blur_limit=3, p=0.2),
    A.RandomGamma(p=0.3),
    A.HueSaturationValue(p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3)
])

# Augment loop
counter = 0
pbar = tqdm(total=needed, desc="Augmenting cocci images")

while counter < needed:
    img_name = random.choice(image_files)
    img_path = os.path.join(INPUT_DIR, img_name)
    img = cv2.imread(img_path)

    if img is None:
        continue

    # Apply augmentation
    augmented = augmentations(image=img)['image']

    # Save augmented image
    save_path = os.path.join(OUTPUT_DIR, f"aug_cocci_{counter+1}.jpg")
    cv2.imwrite(save_path, augmented)
    counter += 1
    pbar.update(1)

pbar.close()
print(f"\n✅ Augmented {needed} images. Total cocci images now: {TARGET_COUNT}")


/tmp/ipykernel_4958/1542313950.py:30: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3)
Augmenting cocci images: 100%|████████████████| 400/400 [00:49<00:00,  8.01it/s]


✅ Augmented 400 images. Total cocci images now: 1250
